In [1]:
import os
import json
from pathlib import Path


root_dirs = ["L5G1_L4G1_L4G3_json", "L4G1_L3G0_L3G1_json", "L4G3_L3G0_L3G2_json"]


target_files = ["group_hierarchy.json", "group_metadata.json", "feature_evolution.json", "feature_weights.json", "group_captions.json"]

for root in root_dirs:
    print(f"\n====== Folder: {root} ======")
    for file_name in target_files:
        file_path = Path(root) / file_name
        if file_path.exists():
            print(f"\n--- File: {file_path} ---")
            with open(file_path, "r", encoding="utf-8") as f:
                try:
                    content = json.load(f)
                    print(json.dumps(content, indent=2, ensure_ascii=False))
                except json.JSONDecodeError as e:
                    print(f"[Error parsing JSON] {e}")
        else:
            print(f"[Missing] {file_path}")



====== Folder: L5G1_L4G1_L4G3_json ======

--- File: L5G1_L4G1_L4G3_json/group_hierarchy.json ---
{
  "L5G1": [
    "L4G1",
    "L4G3"
  ]
}

--- File: L5G1_L4G1_L4G3_json/group_metadata.json ---
{
  "L5G1": {
    "description": "Curved peripheral structures with a gradual gradient of blue to pink",
    "images": [
      "visualizations/layer5_neuron55_group1_label1_img1.png",
      "visualizations/layer5_neuron55_group1_label1_img2.png",
      "visualizations/layer5_neuron55_group1_label1_img3.png",
      "visualizations/layer5_neuron55_group1_label1_img4.png",
      "visualizations/layer5_neuron55_group1_label1_img5.png"
    ],
    "label": 1,
    "feature_map_id": 55
  },
  "L4G1": {
    "description": "Curved edges with a blue to pink gradient",
    "images": [
      "visualizations/layer4_neuron826_group1_label0_img1.png",
      "visualizations/layer4_neuron826_group1_label0_img2.png",
      "visualizations/layer4_neuron826_group1_label0_img3.png",
      "visualizations/layer4_ne

In [2]:
import os
import json
from graphviz import Digraph

# Folders ordered from highest to lowest concept level
folders = [
    "L5G1_L4G1_L4G3_json",
    "L4G1_L3G0_L3G1_json",
    "L4G3_L3G0_L3G2_json"
]

MAX_IMAGES = 5

# Initialize graph
dot = Digraph("ConceptTree", format="png")
dot.attr(rankdir="TB", dpi="150")
dot.attr("node", shape="box", style="filled", fillcolor="#eef6ff", fontname="Helvetica")

node_info = {}
edges = {}

# Load all JSONs from each folder
for folder in folders:
    with open(os.path.join(folder, "group_hierarchy.json")) as f:
        hierarchy = json.load(f)
    with open(os.path.join(folder, "group_metadata.json")) as f:
        metadata = json.load(f)
    with open(os.path.join(folder, "feature_evolution.json")) as f:
        evolution = json.load(f)
    with open(os.path.join(folder, "feature_weights.json")) as f:
        weights = json.load(f)

    # Store node descriptions, image paths, and group label
    for group, info in metadata.items():
        desc = info.get("description", "")
        label_val = info.get("label", None)
        imgs = info.get("images", [])[:MAX_IMAGES]
        node_info[group] = {
            "description": desc,
            "label": label_val,
            "images": imgs
        }

    # Store forward edges with label = evolution + weight
    for parent, children in hierarchy.items():
        for child in children:
            evo = evolution.get(parent, {}).get(child, "")
            weight = weights.get(parent, {}).get(child, None)
            edge_label = evo
            if weight is not None:
                edge_label += f"\n(weight={weight:.2f})"
            edges[(parent, child)] = edge_label  # forward direction

# Add node visuals
for group, info in node_info.items():
    label_parts = [
        f'<TABLE BORDER="0" CELLBORDER="0" CELLSPACING="0" CELLPADDING="4">',
        f'<TR><TD BGCOLOR="#cce6ff" COLSPAN="{MAX_IMAGES}"><B>{group}</B></TD></TR>',
    ]

    # Add label if available
    if info["label"] is not None:
        label_parts.append(f'<TR><TD COLSPAN="{MAX_IMAGES}"><I>label: {info["label"]}</I></TD></TR>')

    # Description
    label_parts.append(f'<TR><TD COLSPAN="{MAX_IMAGES}">{info["description"]}</TD></TR>')

    # Horizontal image row (without illegal attributes)
    img_cells = []
    for img_path in info["images"]:
        full_path = os.path.join("visualizations", os.path.basename(img_path))
        if os.path.exists(full_path):
            img_cells.append(f'<TD><IMG SRC="{full_path}"/></TD>')
    if img_cells:
        label_parts.append(f"<TR>{''.join(img_cells)}</TR>")

    label_parts.append('</TABLE>')
    label = "<" + "\n".join(label_parts) + ">"
    dot.node(group, label=label)

# Add reversed arrows (child → parent) with evolution + weight
for (parent, child), label in edges.items():
    dot.edge(parent, child, label=label, fontname="Helvetica", fontsize="10", dir="back")

# Output as PNG
dot.render("concept_tree", cleanup=True)
print("Saved as concept_tree.png")


Saved as concept_tree.png
